# Первая лабораторная работа: Fusion на датасете PMEmo

Будем решать проблему характеристики песен по тексту и аудио

## Импорты и константы

In [72]:
import re
import warnings
from pathlib import Path
from pprint import pp

import librosa
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from transformers import (
    BertModel,
    BertTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model,
)

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [50]:
SR = 16000
RANDOM_STATE = 42

In [51]:
AUDIO_CORPUS_PATH = Path("data/PMEmo2019/chorus")
LYRICS_CORPUS_PATH = Path("data/PMEmo2019/lyrics")
ANNOTATIONS = Path("data/PMEmo2019/genre_annotations/genre_annotations.csv")

In [52]:
annotations_df = pd.read_csv(ANNOTATIONS, sep=",", index_col="musicId")
annotations_df.head()

,genre
musicId,
1,Hip-Hop/Rap
4,Hip-Hop/Rap
5,Hip-Hop/Rap
6,Hip-Hop/Rap
7,Pop


In [53]:
audio_paths = []
text_paths = []
annotations = []

for root, dirs, files in AUDIO_CORPUS_PATH.walk():
    for file in files:
        audio_path = root / file
        try:
            music_id = int(str(Path(file).with_suffix("")))
            annotation = annotations_df.loc[music_id]
            if isinstance(annotation, pd.DataFrame):
                annotation = annotation.mean()

            text_path = LYRICS_CORPUS_PATH / Path(file).with_suffix(".lrc")
            if not text_path.exists():
                continue

            audio_paths.append(audio_path)
            text_paths.append(text_path)
            annotations.append(annotation.tolist())
        except KeyError:
            pass

In [54]:
print(f"Кол-во аудиозаписей: {len(audio_paths)}\nКол-во лейблов: {len(annotations)}")

Кол-во аудиозаписей: 556
Кол-во лейблов: 556


In [55]:
pp(list(zip(audio_paths[:5], annotations[:5])))

[(PosixPath('data/PMEmo2019/chorus/851.mp3'), ['R&B/Soul']),
 (PosixPath('data/PMEmo2019/chorus/888.mp3'), ['Rock/Alternative']),
 (PosixPath('data/PMEmo2019/chorus/28.mp3'), ['Pop']),
 (PosixPath('data/PMEmo2019/chorus/765.mp3'), ['Rock/Alternative']),
 (PosixPath('data/PMEmo2019/chorus/964.mp3'), ['Hip-Hop/Rap'])]


In [56]:
def read_lrc(text_path):
    def lrc_preprocess(line):
        line = line.strip()
        line = re.sub(r"\[.+\]", "", line)
        return line

    with open(text_path) as f:
        lrc_list = f.readlines()
    lrc_list = [lrc_preprocess(lrc_string) for lrc_string in lrc_list]
    lrc_list = list(filter(len, lrc_list))

    return lrc_list

In [57]:
def make_audio_windows(audio, len_window_ms=30):
    len_window_samples = int(len_window_ms / 1000 * SR)
    windows = []
    for window in range(0, len(audio), len_window_samples):
        if (window + len_window_samples) <= len(audio):
            windows.append(audio[window : window + len_window_samples])
        else:
            windows.append(audio[window:])
    return windows

## Извлечение признаков

### Собственные векторы

In [73]:
def build_feature_vector(audio, n_mfcc=13):
    rms = librosa.feature.rms(y=audio)

    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=SR
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SR,
        n_mfcc=n_mfcc
    )

    mfcc_delta = librosa.feature.delta(mfcc)

    contrast = librosa.feature.spectral_contrast(
        y=audio,
        sr=SR
    )

    flatness = librosa.feature.spectral_flatness(
        y=audio
    )

    def pool(feature):
        return np.concatenate([
            np.mean(feature, axis=1),
            np.std(feature, axis=1)
        ])

    return np.concatenate([
        pool(rms),
        pool(chroma),
        pool(mfcc),
        pool(mfcc_delta),
        pool(contrast),
        pool(flatness)
    ])

In [74]:
audio_feature_vectors = []
for audio_path in tqdm(audio_paths):
    audio, _ = librosa.load(audio_path, sr=SR)
    audio_feature_vectors.append(
        build_feature_vector(audio)
    )

100%|██████████| 556/556 [02:36<00:00,  3.56it/s]


### wav2vec2

In [ ]:
wav2vec2_model = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-base-960h"
)
wav2vec2_model = wav2vec2_model.to(device) # type: ignore
wav2vec2_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-base-960h"
)

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [205]:
wav2vec2_embeddings = []
for audio_path in tqdm(audio_paths):
    audio, _ = librosa.load(audio_path, sr=SR)

    # # -> [[window], [window]]
    # audio = make_audio_windows(audio)

    inputs = wav2vec2_feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    inputs = inputs.to(device)

    with torch.inference_mode():
        outputs = wav2vec2_model(**inputs)

    features = outputs.last_hidden_state
    embedding = features.mean(dim=1).squeeze(0).to("cpu")

    wav2vec2_embeddings.append(embedding)

100%|██████████| 556/556 [05:22<00:00,  1.72it/s]


In [206]:
wav2vec2_embeddings[0].shape

torch.Size([768])

In [207]:
torch.save(wav2vec2_embeddings, "wav2vec2_embeddings.pt")

### BERT

In [208]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

bert_model = bert_model.to(device) # type: ignore

In [ ]:
bert_embeddings = []
for text_path in tqdm(text_paths):
    # -> [[lyrics line], [lyrics line]]
    text = read_lrc(text_path)
    # text = "\n".join(read_lrc(text_path))
    
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        padding=True
    )
    inputs = inputs.to(device)

    with torch.inference_mode():
        outputs = bert_model(
            **inputs, 
            output_hidden_states=True
        )

    features = outputs.last_hidden_state
    embedding = features.mean(dim=0).mean(dim=0).to("cpu")    

    bert_embeddings.append(embedding)

 88%|████████▊ | 490/556 [01:31<00:09,  7.20it/s]

In [ ]:
bert_embeddings[0].shape

torch.Size([768])

In [ ]:
torch.save(bert_embeddings, "bert_embeddings.pt")

## Классификаторы по отдельности

### Аудио свои вектора

In [75]:
feature_vectors_train, feature_vectors_test, \
    annotations_train, annotations_test = train_test_split(
        audio_feature_vectors,
        annotations, 
        test_size=0.2, 
        random_state=RANDOM_STATE
)

In [76]:
rfc = RandomForestClassifier()
params = {
    "n_estimators": [100, 300],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}
grid = GridSearchCV(
    rfc,
    params,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring="f1_macro",
)

In [77]:
grid.fit(feature_vectors_train, annotations_train)
best_estimator = grid.best_estimator_

Fitting 5 folds for each of 64 candidates, totalling 320 fits


In [78]:
rfc_preds = best_estimator.predict(feature_vectors_test)
print(classification_report(annotations_test, rfc_preds))

                  precision    recall  f1-score   support

         Country       0.50      0.74      0.60        19
Dance/Electronic       0.50      0.13      0.21        15
     Hip-Hop/Rap       0.74      0.94      0.83        34
             Pop       0.50      0.48      0.49        25
        R&B/Soul       0.50      0.17      0.25        12
Rock/Alternative       0.11      0.14      0.12         7

        accuracy                           0.56       112
       macro avg       0.48      0.43      0.42       112
    weighted avg       0.55      0.56      0.53       112



### Аудио

In [34]:
wav2vec2_embeddings = torch.load("wav2vec2_embeddings.pt")

In [ ]:
wav2vec2_embeddings_train, wav2vec2_embeddings_test, \
    annotations_train, annotations_test = train_test_split(
        wav2vec2_embeddings,
        annotations, 
        test_size=0.2, 
        random_state=RANDOM_STATE
)

In [ ]:
wav2vec2_embeddings_train[0].shape, annotations_train[0]

(torch.Size([768]), ['Country'])

In [37]:
rfc = RandomForestClassifier()
params = {
    "n_estimators": [100, 300],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}
grid = GridSearchCV(
    rfc,
    params,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring="f1_macro",
)

In [ ]:
grid.fit(wav2vec2_embeddings_train, annotations_train)
best_estimator = grid.best_estimator_

Fitting 5 folds for each of 64 candidates, totalling 320 fits


In [ ]:
rfc_preds = best_estimator.predict(wav2vec2_embeddings_test)
print(classification_report(annotations_test, rfc_preds))

                  precision    recall  f1-score   support

         Country       0.32      0.53      0.40        19
Dance/Electronic       0.25      0.07      0.11        15
     Hip-Hop/Rap       0.69      0.91      0.78        34
             Pop       0.22      0.16      0.19        25
        R&B/Soul       0.14      0.08      0.11        12
Rock/Alternative       0.00      0.00      0.00         7

        accuracy                           0.42       112
       macro avg       0.27      0.29      0.26       112
    weighted avg       0.36      0.42      0.37       112



### Текст

In [40]:
bert_embeddings = torch.load("bert_embeddings.pt")

In [41]:
bert_embeddings_train, bert_embeddings_test, \
    bert_annotations_train, bert_annotations_test = train_test_split(
        bert_embeddings,
        annotations, 
        test_size=0.2, 
        random_state=RANDOM_STATE
)

In [42]:
bert_embeddings_train[0].shape, bert_annotations_train[0]

(torch.Size([768]), ['Country'])

In [43]:
rfc = RandomForestClassifier()
params = {
    "n_estimators": [100, 300],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}
grid = GridSearchCV(
    rfc,
    params,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring="f1_macro",
)

In [44]:
grid.fit(bert_embeddings_train, bert_annotations_train)
best_estimator = grid.best_estimator_

Fitting 5 folds for each of 64 candidates, totalling 320 fits


In [45]:
rfc_preds = best_estimator.predict(bert_embeddings_test)
print(classification_report(bert_annotations_test, rfc_preds))

                  precision    recall  f1-score   support

         Country       0.43      0.63      0.51        19
Dance/Electronic       0.50      0.27      0.35        15
     Hip-Hop/Rap       0.78      0.85      0.82        34
             Pop       0.39      0.48      0.43        25
        R&B/Soul       0.00      0.00      0.00        12
Rock/Alternative       0.29      0.29      0.29         7

        accuracy                           0.53       112
       macro avg       0.40      0.42      0.40       112
    weighted avg       0.48      0.53      0.49       112



## Early Fusion

In [65]:
wav2vec2_embeddings = torch.load("wav2vec2_embeddings.pt")
bert_embeddings = torch.load("bert_embeddings.pt")

early_fused_embs = []
for wav2vec2, bert in zip(wav2vec2_embeddings, bert_embeddings):
    early_fused_embs.append(
        torch.cat((wav2vec2, bert))
    )

In [66]:
len(early_fused_embs), len(annotations)

(556, 556)

In [67]:
early_fused_embs_train, early_fused_embs_test, \
    annotations_train, annotations_test = train_test_split(
        early_fused_embs,
        annotations
    )

In [68]:
scaler = StandardScaler()
early_fused_embs_train = scaler.fit_transform(early_fused_embs_train)
early_fused_embs_test = scaler.transform(early_fused_embs_test)

In [69]:
rfc = RandomForestClassifier()
params = {
    "n_estimators": [100, 300],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"],
}
grid = GridSearchCV(
    rfc,
    params,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring="f1_macro",
)

In [70]:
grid.fit(early_fused_embs_train, annotations_train)
best_estimator = grid.best_estimator_

Fitting 5 folds for each of 64 candidates, totalling 320 fits


In [71]:
rfc_preds = best_estimator.predict(early_fused_embs_test)
print(classification_report(annotations_test, rfc_preds))

                  precision    recall  f1-score   support

         Country       0.52      0.43      0.47        28
Dance/Electronic       0.00      0.00      0.00        10
     Hip-Hop/Rap       0.70      0.76      0.73        41
             Pop       0.30      0.40      0.34        30
        R&B/Soul       0.06      0.07      0.06        15
Rock/Alternative       0.33      0.20      0.25        15

        accuracy                           0.42       139
       macro avg       0.32      0.31      0.31       139
    weighted avg       0.42      0.42      0.42       139



## Intermideate Fusion

## Late Fusion